# Modelado de Recontacto - XGBoost (Adaptación)
En lugar de predecir el abandono de vuelos, predeciremos si un ciudadano **volverá a contactar (recontacto_7_dias)**.

In [ ]:
import pyodbc
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier

# Cargar datos
conn = pyodbc.connect(f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER=db;DATABASE=ServicioCiudadanoDW;UID=SA;PWD=J0AhMXX4H0QBrRr8R1U098y8aA1@;TrustServerCertificate=yes;")
df = pd.read_sql("SELECT * FROM vw_v1_limpio", conn)
conn.close()

# Preparar variables
X = df.drop(columns=['id_interaccion', 'fecha_contacto', 'fecha_carga', 'recontacto_7_dias'])
y = df['recontacto_7_dias']

# One-hot encoding de las categóricas
columnas_texto = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=columnas_texto.tolist(), drop_first=True)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"✅ Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")


In [ ]:
# Entrenar el modelo (XGBoost)
# Balanceo de clases (scale_pos_weight) es vital porque hay menos recontactos que resueltos
ratio_desbalance = (y_train == 0).sum() / (y_train == 1).sum()

modelo = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=ratio_desbalance, 
    random_state=42,
    eval_metric='auc'
)

print("🔄 Entrenando modelo XGBoost...")
modelo.fit(X_train, y_train)
print("✅ Modelo entrenado exitosamente")

# Evaluacion
y_pred = modelo.predict(X_test)
y_pred_proba = modelo.predict_proba(X_test)[:, 1]

print("=" * 60)
print("📊 REPORTE DE CLASIFICACIÓN (Recontacto)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Resuelto (0)', 'Recontacta (1)']))
print(f"🎯 ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")


In [ ]:
# Gráfico de Matriz de Confusión y Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Matriz
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Resuelto (0)', 'Recontacta (1)'], 
            yticklabels=['Resuelto (0)', 'Recontacta (1)'], ax=axes[0])
axes[0].set_title('Matriz de Confusión')
axes[0].set_ylabel('Valor Real')
axes[0].set_xlabel('Predicción')

# Importancia
importancia = pd.DataFrame({'Variable': X.columns, 'Importancia': modelo.feature_importances_})
importancia = importancia.sort_values(by='Importancia', ascending=False).head(10)
sns.barplot(x='Importancia', y='Variable', data=importancia, palette='viridis', ax=axes[1])
axes[1].set_title('Top 10 Variables que Predicen Recontacto')

plt.tight_layout()
plt.show()
